In [1]:
import duckdb
from networks.cnn_no_local_pooling import CNNModel
from networks.cnn_network import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 632,930
  -> ny bästa modell sparad till ../models/cnn_no_local_pooling.pth
Epoch   0 | train: 2.8223 | val: 0.7526 | acc: 4.23% | AUC: 0.500  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_no_local_pooling.pth
Epoch   1 | train: 0.6979 | val: 0.7377 | acc: 4.23% | AUC: 0.504  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_no_local_pooling.pth
Epoch   2 | train: 0.6960 | val: 0.7177 | acc: 4.23% | AUC: 0.532  | LR: 0.001
Epoch   3 | train: 0.6762 | val: 0.7058 | acc: 4.23% | AUC: 0.511  | LR: 0.001
Epoch   4 | train: 0.6902 | val: 0.6939 | acc: 4.23% | AUC: 0.511  | LR: 0.001
Epoch   5 | train: 0.6825 | val: 0.6802 | acc: 95.95% | AUC: 0.522  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_no_local_pooling.pth
Epoch   6 | train: 0.6584 | val: 0.6507 | acc: 90.20% | AUC: 0.679  | LR: 0.001
Epoch   7 | train: 0.6256 | val: 0.6152 | acc: 96.96% | A

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.121852,0.199031,94.707521,0.977092,0.954377,0.895911,1799,86,28,241
1,2,0.056426,0.132752,96.378830,0.986126,0.966048,0.947955,1821,64,14,255
2,3,0.095675,0.099349,98.050139,0.991952,0.985676,0.944238,1858,27,15,254
3,4,0.100718,0.131739,97.165428,0.988297,0.979288,0.918216,1844,39,22,247
4,5,0.118998,0.167731,96.005574,0.984031,0.968153,0.903346,1824,60,26,243
5,6,0.123459,0.420459,97.446611,0.980511,0.982493,0.918216,1852,33,22,247
6,7,0.057557,0.323113,95.680446,0.981101,0.964968,0.899628,1818,66,27,242
7,8,0.210302,0.386982,90.343547,0.924864,0.918302,0.799257,1731,154,54,215
8,9,0.092703,0.146815,97.538319,0.988824,0.984599,0.911111,1854,29,24,246
9,10,0.114665,0.152552,98.327138,0.994532,0.995218,0.900000,1873,9,27,243


In [1]:
import duckdb
from networks.cnn_batchnorm1d import CNNModel
from networks.cnn_network import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 633,122
  -> ny bästa modell sparad till ../models/cnn_batchnorm.pth
Epoch   0 | train: 0.6081 | val: 0.4717 | acc: 91.72% | AUC: 0.921  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_batchnorm.pth
Epoch   1 | train: 0.3968 | val: 0.2797 | acc: 96.23% | AUC: 0.960  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_batchnorm.pth
Epoch   2 | train: 0.2897 | val: 0.1479 | acc: 98.33% | AUC: 0.985  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_batchnorm.pth
Epoch   3 | train: 0.1982 | val: 0.0980 | acc: 98.75% | AUC: 0.991  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_batchnorm.pth
Epoch   4 | train: 0.1938 | val: 0.1199 | acc: 97.72% | AUC: 0.995  | LR: 0.001
Epoch   5 | train: 0.1563 | val: 0.0827 | acc: 98.63% | AUC: 0.995  | LR: 0.001
Epoch   6 | train: 0.1522 | val: 0.0866 | acc: 98.51% | AUC: 0.995  | LR: 0.001
  -> ny bästa model

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.057810,0.135714,97.352531,0.992144,0.978757,0.937037,1843,40,17,253
1,2,0.053361,0.103239,96.470042,0.992551,0.968136,0.940741,1823,60,16,254
2,3,0.065466,0.139423,96.843083,0.989629,0.975597,0.918216,1839,46,22,247
3,4,0.074243,0.161690,97.956340,0.987948,0.987261,0.925651,1860,24,20,249
4,5,0.051941,0.098871,97.165428,0.994727,0.973978,0.955390,1834,49,12,257
5,6,0.050924,0.105572,97.724106,0.992978,0.983546,0.933086,1853,31,18,251
6,7,0.052010,0.092802,97.584765,0.994189,0.979299,0.951673,1845,39,13,256
7,8,0.090618,0.155616,96.003717,0.987926,0.964418,0.929368,1816,67,19,250
8,9,0.129748,0.198991,97.213191,0.982460,0.983546,0.892193,1853,31,29,240
9,10,0.097127,0.146258,97.539461,0.990575,0.988859,0.881041,1864,21,32,237


In [1]:
import duckdb
from networks.cnn_batchnorm_and_local_pool import CNNModel
from networks.cnn_network import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 172,322
  -> ny bästa modell sparad till ../models/cnn_batchnorm_and_local_pool.pth
Epoch   0 | train: 0.5048 | val: 0.2750 | acc: 97.38% | AUC: 0.935  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_batchnorm_and_local_pool.pth
Epoch   1 | train: 0.3000 | val: 0.1658 | acc: 97.72% | AUC: 0.970  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_batchnorm_and_local_pool.pth
Epoch   2 | train: 0.2290 | val: 0.1125 | acc: 98.45% | AUC: 0.992  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_batchnorm_and_local_pool.pth
Epoch   3 | train: 0.2085 | val: 0.1257 | acc: 97.05% | AUC: 0.994  | LR: 0.001
Epoch   4 | train: 0.1782 | val: 0.1027 | acc: 98.42% | AUC: 0.993  | LR: 0.001
Epoch   5 | train: 0.1639 | val: 0.4116 | acc: 79.40% | AUC: 0.987  | LR: 0.001
Epoch   6 | train: 0.1451 | val: 0.1353 | acc: 95.65% | AUC: 0.993  | LR: 0.001
  -> ny bästa modell

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.044997,0.140434,97.259638,0.991165,0.977695,0.937037,1841,42,17,253
1,2,0.080701,0.098901,97.353760,0.992514,0.978769,0.937037,1844,40,17,253
2,3,0.069863,0.138811,96.889508,0.987225,0.976127,0.918216,1840,45,22,247
3,4,0.053628,0.147692,96.837209,0.988713,0.971292,0.947955,1827,54,14,255
4,5,0.049181,0.114955,97.632312,0.992082,0.980371,0.947955,1848,37,14,255
5,6,0.061636,0.102457,97.724106,0.991697,0.981423,0.947955,1849,35,14,255
6,7,0.092513,0.105932,97.721990,0.990783,0.983528,0.933086,1851,31,18,251
7,8,0.059190,0.157045,96.471681,0.985771,0.970292,0.925651,1829,56,20,249
8,9,0.102520,0.185383,95.264624,0.982069,0.953316,0.947955,1797,88,14,255
9,10,0.234930,0.250878,95.218199,0.987988,0.950133,0.966543,1791,94,9,260
